In [1]:
# Load env variables and create client
import base64
from dotenv import load_dotenv
from anthropic import Anthropic

load_dotenv()

client = Anthropic()
model = "claude-sonnet-4-5"

In [2]:
# Helper functions
from anthropic.types import Message


def add_user_message(messages, message):
    user_message = {
        "role": "user",
        "content": message.content if isinstance(message, Message) else message,
    }
    messages.append(user_message)


def add_assistant_message(messages, message):
    assistant_message = {
        "role": "assistant",
        "content": message.content if isinstance(message, Message) else message,
    }
    messages.append(assistant_message)


def chat(
    messages,
    system=None,
    temperature=1.0,
    stop_sequences=[],
    tools=None,
    thinking=False,
    thinking_budget=1024,
):
    params = {
        "model": model,
        "max_tokens": 4000,
        "messages": messages,
        "temperature": temperature,
        "stop_sequences": stop_sequences,
    }

    if thinking:
        params["thinking"] = {
            "type": "enabled",
            "budget_tokens": thinking_budget,
        }

    if tools:
        params["tools"] = tools

    if system:
        params["system"] = system

    message = client.messages.create(**params)
    return message


def text_from_message(message):
    return "\n".join([block.text for block in message.content if block.type == "text"])

In [ ]:
# TODO: Read pdf, feed into Claude
# Almost identical code to what you'd use for images
# To send an image to Claude, include an image block in the user message alongside text blocks

with open("earth.pdf", "rb") as f:
    file_bytes = base64.standard_b64encode(f.read()).decode("utf-8") # images can be included as base64 encoding or a url to the image

messages = []
add_user_message(messages, [
    # Image Block
    {
        "type": "document", #Change here on file type
        "source": {
            "type": "base64",
            "media_type": "application/pdf", # Change here on file type
            "data": file_bytes
        }
    },
    # Text Block
    {
        "type": "text",
        "text": 'How were Earths atmosphere and oceans formed?'
    }
]
)

In [6]:
# Call the chat function
chat(messages)

Message(id='msg_01QucA7qZ5gfhrwRk7aQC1rt', container=None, content=[TextBlock(citations=None, text="# Formation of Earth's Atmosphere and Oceans\n\nAccording to the document, Earth's atmosphere and oceans were formed through the following process:\n\n## Primary Formation Mechanism\n**Volcanic activity and outgassing** - The atmosphere and oceans were created by gases and water vapor released from Earth's interior through volcanic processes.\n\n## Ocean Formation\nThe water vapor from volcanic outgassing **condensed into the oceans**. This water was then augmented (added to) by additional sources:\n- Water and ice from **asteroids**\n- Water from **protoplanets**\n- Water from **comets**\n\n## Timeline and Conditions\n- Sufficient water to fill the oceans may have been present on Earth **since it formed** (around 4.54 billion years ago)\n- Atmospheric **greenhouse gases** played a crucial role in keeping the oceans from freezing during early Earth history, when the newly forming Sun had